# 第 14 章 · launch 进阶、测试与调试

本章回答：真实机器人的 launch 文件动辄几百行，怎么组织？参数、条件、"某个节点挂了就全关"怎么写？ROS 2 程序怎么写自动测试？节点崩了怎么拿到崩溃位置？回调到底花了多久怎么量？

配套讲义：仓库里 `ros2/notes/lessons/14-launch-test-debug.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 launch / launch_ros。这里用几个小类模仿 LaunchConfiguration、PathJoinSubstitution、PythonExpression、PushRosNamespace：写的时候只是占位，launch() 时才按命令行参数和默认值算出真值。
条件（IfCondition）和事件处理器留给练习 14-1、14-2，这里不演示。

In [ ]:
print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 总图 描述不是脚本 | 写下来的是占位 | `type(description[0]['namespace'])` | `LaunchConfiguration` |
| 第 1 步 默认值 | `default_value='turtlesim1'` | `launch(description, declared, {})` | `/turtlesim1/sim` |
| 第 1 步 命令行传参 | `turtlesim_ns:=robot7` | `launch(..., {'turtlesim_ns': 'robot7'})` | `/robot7/sim` |
| 第 1 步 Include 传参 | `launch_arguments` | 同上传 turtlesim2 | `/turtlesim2/sim` |
| 第 1 步 PushRosNamespace | 整组再套一层 | `launch(..., namespace='robot1')` | `/robot1/turtlesim1/sim` |
| 第 1 步 其他替换 | PathJoin、PythonExpression | `launch([node], ...)` | 路径拼好；`0.5 * 2` → 1.0 |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**类 `LaunchConfiguration`**：占位：launch 参数 name 的值，启动时才知道。

In [ ]:
class LaunchConfiguration:
    """占位：launch 参数 name 的值，启动时才知道。"""

    def __init__(self, name: str):
        self.name = name

    def perform(self, ctx: dict) -> str:
        return ctx[self.name]

**函数 `resolve`**：字符串原样，占位就调 perform。

In [ ]:
def resolve(value, ctx: dict) -> str:
    """字符串原样，占位就调 perform。"""
    return value.perform(ctx) if hasattr(value, "perform") else value

**类 `PathJoinSubstitution`**：把几段（可以含占位）拼成路径。

In [ ]:
class PathJoinSubstitution:
    """把几段（可以含占位）拼成路径。"""

    def __init__(self, parts: list):
        self.parts = parts

    def perform(self, ctx: dict) -> str:
        return "/".join(resolve(p, ctx) for p in self.parts)

**类 `PythonExpression`**：启动时把几段拼成一个 Python 表达式再求值。

In [ ]:
class PythonExpression:
    """启动时把几段拼成一个 Python 表达式再求值。"""

    def __init__(self, parts: list):
        self.parts = parts

    def perform(self, ctx: dict) -> str:
        return str(eval("".join(resolve(p, ctx) for p in self.parts), {}, {}))

**函数 `launch`**：ros2 launch：先定参数（命令行 > 默认值），再把每个 Node 的占位全部算出来。返回每个节点的完整名字。

In [ ]:
def launch(description: list[dict], declared: dict[str, str], cli: dict[str, str], namespace: str = "") -> list[str]:
    """ros2 launch：先定参数（命令行 > 默认值），再把每个 Node 的占位全部算出来。返回每个节点的完整名字。"""
    ctx = {**declared, **cli}
    out = []
    for node in description:
        ns = "/".join(s for s in [namespace.strip("/"), resolve(node.get("namespace", ""), ctx).strip("/")] if s)
        extra = {k: resolve(v, ctx) for k, v in node.items() if k not in ("namespace", "name")}
        out.append(f"/{ns}/{node['name']}" if ns else f"/{node['name']}")
        if extra:
            out[-1] += f"  {extra}"
    return out

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
turtlesim_ns = LaunchConfiguration("turtlesim_ns")
description = [{"namespace": turtlesim_ns, "name": "sim"}]  # Node(namespace=turtlesim_ns, name='sim')
declared = {"turtlesim_ns": "turtlesim1"}  # DeclareLaunchArgument('turtlesim_ns', default_value='turtlesim1')

In [ ]:
print("\n总图 · 写下来的时候只是占位，不是值")
print("  description 里的 namespace 是:", type(description[0]["namespace"]).__name__)

In [ ]:
print("\n第一步 · 启动时才代入：默认值 vs 命令行 turtlesim_ns:=robot7")
print("  默认:", launch(description, declared, {}))
print("  turtlesim_ns:=robot7:", launch(description, declared, {"turtlesim_ns": "robot7"}))

In [ ]:
print("\n第一步（续）· 包含另一个 launch 并传参：launch_arguments={'turtlesim_ns': 'turtlesim2'}")
print("  ", launch(description, declared, {"turtlesim_ns": "turtlesim2"}))

In [ ]:
print("\n第一步（再续）· GroupAction + PushRosNamespace('robot1')：整组再套一层命名空间")
print("  ", launch(description, declared, {}, namespace="robot1"))

In [ ]:
print("\n第一步（又续）· PathJoinSubstitution 和 PythonExpression 也是启动时才算")
node = {"name": "loader",
        "params_file": PathJoinSubstitution(["/opt/ros/lyrical/share", LaunchConfiguration("pkg"), "config", "a.yaml"]),
        "scale": PythonExpression([LaunchConfiguration("base_speed"), " * 2"])}  # 参数名和值是编的
print("  ", launch([node], {"pkg": "my_robot", "base_speed": "0.5"}, {}))

## 3. 代码里才看得到的

1. **`ctx = {**declared, **cli}`**：后面的字典覆盖前面的，所以命令行的值优先于默认值。
2. **PythonExpression 用的是 `eval`**：真 launch 也是把几段拼成字符串再求值，所以字符串里的引号、空格都要自己写对。
3. **条件（IfCondition）和事件处理器没有模拟**，留给练习 14-1、14-2。

## 4. 自己改一改

- 在 `description` 里再加一个 Node（比如 `name='teleop'`，同一个 namespace），看两个节点的名字；
- 把 `base_speed` 改成 `'0.3'`，看 `scale` 变成多少。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **launch 是描述不是脚本**：替换启动时才求值，判断要用 `IfCondition`。
- **大项目**：顶层只做 include，参数往下传，配置进 YAML，命名空间分组。
- **事件**：关键进程退出就 Shutdown，别让系统半残。
- **测试三层**：pytest 测函数，launch_testing 测节点，ament_lint 查风格；`colcon test` 一起跑。
- **调试**：崩溃 gdb 看栈，行为看日志，耗时用 tracing。